# Setup OpenSearch Serverless Collection

This notebook will guide you through setting up an OpenSearch Serverless collection for vector search.

## 1. Import Required Libraries

In [ ]:
import boto3
import json
import time
from datetime import datetime

## 2. Configure AWS Credentials

In [ ]:
# Set your AWS region
AWS_REGION = 'us-west-2'

# Initialize clients
opensearch_client = boto3.client('opensearchserverless', region_name=AWS_REGION)
sts_client = boto3.client('sts', region_name=AWS_REGION)

# Get account information
account_id = sts_client.get_caller_identity()['Account']
print(f"AWS Account ID: {account_id}")
print(f"Region: {AWS_REGION}")

## 3. Create Security Policies

In [ ]:
collection_name = 'movie-search'

# Encryption Policy
encryption_policy_name = f"{collection_name}-encrypt"
encryption_policy = {
    "Rules": [
        {
            "ResourceType": "collection",
            "Resource": [f"collection/{collection_name}"]
        }
    ],
    "AWSOwnedKey": True
}

try:
    opensearch_client.create_security_policy(
        name=encryption_policy_name,
        type='encryption',
        policy=json.dumps(encryption_policy)
    )
    print(f"✓ Created encryption policy: {encryption_policy_name}")
except opensearch_client.exceptions.ConflictException:
    print(f"✓ Encryption policy already exists: {encryption_policy_name}")

In [ ]:
# Network Policy
network_policy_name = f"{collection_name}-network"
network_policy = [
    {
        "Rules": [
            {
                "ResourceType": "collection",
                "Resource": [f"collection/{collection_name}"]
            },
            {
                "ResourceType": "dashboard",
                "Resource": [f"collection/{collection_name}"]
            }
        ],
        "AllowFromPublic": True
    }
]

try:
    opensearch_client.create_security_policy(
        name=network_policy_name,
        type='network',
        policy=json.dumps(network_policy)
    )
    print(f"✓ Created network policy: {network_policy_name}")
except opensearch_client.exceptions.ConflictException:
    print(f"✓ Network policy already exists: {network_policy_name}")

In [ ]:
# Data Access Policy
data_policy_name = f"{collection_name}-data"
caller_identity = sts_client.get_caller_identity()
principal_arn = caller_identity['Arn']

data_policy = [
    {
        "Rules": [
            {
                "ResourceType": "collection",
                "Resource": [f"collection/{collection_name}"],
                "Permission": [
                    "aoss:CreateCollectionItems",
                    "aoss:DeleteCollectionItems",
                    "aoss:UpdateCollectionItems",
                    "aoss:DescribeCollectionItems"
                ]
            },
            {
                "ResourceType": "index",
                "Resource": [f"index/{collection_name}/*"],
                "Permission": [
                    "aoss:CreateIndex",
                    "aoss:DeleteIndex",
                    "aoss:UpdateIndex",
                    "aoss:DescribeIndex",
                    "aoss:ReadDocument",
                    "aoss:WriteDocument"
                ]
            }
        ],
        "Principal": [principal_arn]
    }
]

try:
    opensearch_client.create_access_policy(
        name=data_policy_name,
        type='data',
        policy=json.dumps(data_policy)
    )
    print(f"✓ Created data access policy: {data_policy_name}")
    print(f"  Principal: {principal_arn}")
except opensearch_client.exceptions.ConflictException:
    print(f"✓ Data access policy already exists: {data_policy_name}")

## 4. Create OpenSearch Serverless Collection

In [ ]:
# Wait for policies to propagate
time.sleep(2)

try:
    response = opensearch_client.create_collection(
        name=collection_name,
        type='VECTORSEARCH',
        description='Movie search with semantic vector search'
    )
    collection_id = response['createCollectionDetail']['id']
    print(f"✓ Created collection: {collection_name}")
    print(f"  Collection ID: {collection_id}")
    print(f"  Status: {response['createCollectionDetail']['status']}")
except opensearch_client.exceptions.ConflictException:
    print(f"✓ Collection already exists: {collection_name}")
    response = opensearch_client.batch_get_collection(names=[collection_name])
    collection_id = response['collectionDetails'][0]['id']

## 5. Wait for Collection to Become Active

In [ ]:
print("Waiting for collection to become active...")

while True:
    status_response = opensearch_client.batch_get_collection(names=[collection_name])
    if status_response['collectionDetails']:
        status = status_response['collectionDetails'][0]['status']
        print(f"  Current status: {status}")
        
        if status == 'ACTIVE':
            endpoint = status_response['collectionDetails'][0]['collectionEndpoint']
            dashboard_endpoint = status_response['collectionDetails'][0].get('dashboardEndpoint', 'N/A')
            
            print(f"\n✅ Collection is ACTIVE!")
            print(f"  Collection Endpoint: {endpoint}")
            print(f"  Dashboard Endpoint: {dashboard_endpoint}")
            break
        elif status == 'FAILED':
            print(f"❌ Collection creation failed!")
            break
    
    time.sleep(10)

## 6. Save Configuration

In [ ]:
config = {
    "collection_name": collection_name,
    "collection_id": collection_id,
    "endpoint": endpoint,
    "dashboard_endpoint": dashboard_endpoint,
    "region": AWS_REGION,
    "created_at": datetime.now().isoformat()
}

with open('../config.json', 'w') as f:
    json.dump(config, f, indent=2)

print("✓ Configuration saved to config.json")
print("\n" + "="*70)
print("✅ Setup Complete!")
print("="*70)
print(f"\nNext step: Run notebook 02_Index_Movies.ipynb")